### Package Installations

In [8]:
# Install all dependencies
%pip install "numpy<2" medspacy spacy openai pandas textstat rouge-score
%pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

# Download the base SpaCy model
import spacy
spacy.cli.download("en_core_web_sm")
print("All installations complete and in the correct environment!")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz (119.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
All installations complete and in the correct environment!


### Imports and LLM Initialization

In [19]:
import pandas as pd
import medspacy
import textstat
from rouge_score import rouge_scorer
from openai import OpenAI
import re
import ast
import warnings
warnings.filterwarnings('ignore')

print("Loading MedSpaCy model...")
nlp = medspacy.load()

# --- EVALUATOR API SETUP ---
print("\n" + "="*50)
print("  CLOUD LLM SETUP ")
print("="*50)

# Evaluator: Paste your Groq API Key here
GROQ_API_KEY = "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"

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY
)

if GROQ_API_KEY == "PASTE_YOUR_KEY_HERE":
    print("\n⚠️ WARNING: Please update the GROQ_API_KEY variable with a valid key.")
else:
    print("\nSuccess! Environment setup complete.")

Loading MedSpaCy model...

  CLOUD LLM SETUP 

Success! Environment setup complete.


### Step 1: Data Ingestion and Filtering

In [10]:
import pandas as pd

notes_path = "../data/NOTEEVENTS_sorted.csv"
patients_path = "../data/PATIENTS_sorted.csv"

print("Attempting to load data from exact path...")
notes = pd.read_csv(notes_path, nrows=5000)
patients = pd.read_csv(patients_path)

df = pd.merge(notes, patients[['SUBJECT_ID', 'DOB']], on='SUBJECT_ID')
elderly_notes = df[df['CATEGORY'] == 'Discharge summary'].head(100)
test_note = elderly_notes.iloc[0]['TEXT']

print("Success! Data loaded.")

Attempting to load data from exact path...
Success! Data loaded.


### Step 2: Pipeline Function Definitions

In [17]:
def generate_patient_summary(raw_text, locked_facts):
    """Stage 2: Fact-Anchored Generation for 8th Grade Target"""

    system_prompt = f"""You are a caregiver for an elderly patient.
Your goal is to explain their medical notes and give clear instructions using very simple words.
Target a 6th to 8th-grade reading level.

STRICT FORMATTING RULES:
1. Use ONLY these three headings: "Diagnosis:", "Medication:", "Red Flags:".
2. "Diagnosis:" MUST be a short, reassuring paragraph. Do NOT use bullet points here.
3. "Medication:" MUST be a bulleted list.
4. "Red Flags:" MUST always be generated with 2 or 3 warning signs.

STRICT CONTENT RULES (For Readability):
1. SYLLABLE REDUCTION: Use words like 'Pills' or 'Meds' instead of 'Medication'.
2. STRIP SUFFIXES: Remove 'Sodium', 'HCl', 'Calcium', etc., from drug names.
3. INSTRUCTIONAL VOICE: Use phrases like "Take as prescribed" combined with frequency and reason.

STRICT ALIGNMENT RULES (For ROUGE-L):
1. FACT INCLUSION: You MUST include every term from this list: {locked_facts}
2. PARAGRAPH ANCHORING: State the EXACT medical term first, then explain it in the NEXT sentence. Never place explanations in parentheses immediately after a term.
   Example: "You were treated for Coronary artery disease. This means some of the vessels that bring blood to your heart were blocked."
3. ORDERING: Present conditions in the Diagnosis paragraph in the SAME ORDER they appear in the locked facts list: {locked_facts}
4. BULLET ANCHORING: Start each medication bullet with the EXACT medical term, followed by a colon.
   Example: "* Aspirin: Take as prescribed every day to help prevent blood clots."

LINGUISTIC CLEANUP:
1. Do NOT write "by mouth". Change 'TID' to '3 times a day', 'QID' to '4 times a day', 'PRN' to 'as needed'.
2. Use 'Doctor' instead of 'Physician' or 'PCP'.
3. Ignore all bracketed data like [** **]."""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt + " List medications clearly using bullet points to ensure accuracy."},
            {"role": "user", "content": raw_text}
        ],
        temperature=0  # Zero temperature for fully deterministic output
    )
    return response.choices[0].message.content

### Step 3: Automated Evaluation Metrics

In [21]:
import textstat
from rouge_score import rouge_scorer


def fact_recall(summary, facts_string):
    """Fraction of locked facts that appear verbatim in the summary."""
    summary_lower = summary.lower()
    fact_list = [f.strip() for f in facts_string.split(",") if f.strip()]
    hits = sum(1 for f in fact_list if f.lower() in summary_lower)
    return hits / len(fact_list) if fact_list else 0.0


def evaluate_summary(summary, original_facts):
    """Calculates Readability and Clinical Fidelity metrics."""

    if isinstance(original_facts, list):
        original_facts = " ".join(original_facts)

    # 1. Flesch-Kincaid Grade Level (target: <= 8.0)
    fk_grade = textstat.flesch_kincaid_grade(summary)

    # 2. ROUGE-L recall
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_l = scorer.score(original_facts, summary)['rougeL'].recall

    # 3. Fact Recall (verbatim term matching)
    fr = fact_recall(summary, original_facts)

    return fk_grade, rouge_l, fr

### Step 4: Load NLP Model with Clinical Facts

In [22]:
import medspacy
import en_ner_bc5cdr_md

print("Loading Clinical NLP Model...")

# Disable the base model's parser so it doesn't conflict with MedSpaCy
base_clinical_nlp = en_ner_bc5cdr_md.load(disable=["parser"])

# Pass that loaded model into MedSpaCy
nlp = medspacy.load(base_clinical_nlp)


def extract_clinical_facts(raw_text):
    """Stage 1: Upgraded ScispaCy/MedSpaCy NER Extraction"""
    doc = nlp(raw_text)

    # The clinical model uses 'CHEMICAL' (for drugs) and 'DISEASE'
    target_labels = ['CHEMICAL', 'DISEASE']

    # Extract entities and remove duplicates
    entities = sorted(set([ent.text for ent in doc.ents if ent.label_ in target_labels]))

    # Join them into a comma-separated string
    return ", ".join(entities)


# Test it right away!
test_note = """
Discharge Diagnosis: Community-acquired pneumonia. Right lower zone consolidation.
Medications on Discharge: Amoxicillin 500mg PO TID for 7 days. Ibuprofen 400mg PO PRN for fever or pain.
"""
print("EXTRACTED FACTS:", extract_clinical_facts(test_note))

Loading Clinical NLP Model...


2026-05-12 20:53:32.940 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=7] [doc 0] Token 1 'Discharge' marked as sentence start (span begin)
2026-05-12 20:53:32.940 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=7] [doc 0] Token 9 'Right' marked as sentence start (span end next token)
2026-05-12 20:53:32.940 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=7] [doc 0] Token 9 'Right' marked as sentence start (span begin)
2026-05-12 20:53:32.940 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=7] [doc 0] Token 14 '
' marked as sentence start (span end whitespace)
2026-05-12 20:53:32.940 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=7] [doc 0] GAP DETECTED: tokens 14-14 (idx 83-83) between spans 83-84
2026-05-12 20:53:32.940 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=7] [doc 0] Token 14 '

EXTRACTED FACTS: Amoxicillin, Ibuprofen, fever, pain, pneumonia


### Step 5: End-to-End Pipeline Execution

In [23]:
import pandas as pd
import re


def clean_mimic_text(raw_text):
    """Removes MIMIC-III de-identification brackets."""
    cleaned = re.sub(r'\[\*\*.*?\*\*\]', '[Redacted]', raw_text)
    cleaned = re.sub(r'\[.*?\]', '', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned


print("Loading datasets...")
df_notes    = pd.read_csv("../data/NOTEEVENTS_sorted.csv")
df_patients = pd.read_csv("../data/PATIENTS_sorted.csv")

df_notes['CHARTDATE']  = pd.to_datetime(df_notes['CHARTDATE'])
df_patients['DOB']     = pd.to_datetime(df_patients['DOB'])
df_merged = df_notes.merge(df_patients[['SUBJECT_ID', 'DOB']], on='SUBJECT_ID', how='left')
df_merged['AGE_AT_NOTE'] = (df_merged['CHARTDATE'].dt.year - df_merged['DOB'].dt.year)

elderly_summaries = df_merged[
    (df_merged['CATEGORY'] == 'Discharge summary') &
    (df_merged['AGE_AT_NOTE'] >= 65)
]

if elderly_summaries.empty:
    print("Error: No elderly patients with discharge summaries found.")
else:
    random_record      = elderly_summaries.sample(n=1, random_state=42).iloc[0]
    TARGET_PATIENT_ID  = int(random_record['SUBJECT_ID'])
    patient_age        = int(random_record['AGE_AT_NOTE'])
    raw_clinical_note  = random_record['TEXT']
    real_clinical_note = clean_mimic_text(raw_clinical_note)

    print(f"\n=====================================")
    print(f" PROCESSING RANDOM ELDERLY PATIENT: {TARGET_PATIENT_ID}")
    print(f" PATIENT AGE: {patient_age if patient_age < 150 else '89+'}")
    print(f"=====================================")

    print("\n--- STAGE 1: EXTRACTING FACTS ---")
    facts = extract_clinical_facts(real_clinical_note)
    print(f"Locked Facts: {facts}")

    print("\n--- STAGE 2: GENERATING SUMMARY ---")
    summary = generate_patient_summary(real_clinical_note, facts)
    print(f"Final Summary:\n{summary}")

    print("\n--- STAGE 3: EVALUATION METRICS ---")
    fk, rouge, fr = evaluate_summary(summary, facts)
    print(f"  Flesch-Kincaid Grade : {fk:.2f}   (Target: <= 8.0)")
    print(f"  ROUGE-L Recall       : {rouge:.3f}")
    print(f"  Fact Recall          : {fr:.3f}")

Loading datasets...

 PROCESSING RANDOM ELDERLY PATIENT: 4490
 PATIENT AGE: 83

--- STAGE 1: EXTRACTING FACTS ---


2026-05-12 20:53:37.996 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=8] [doc 0] Token 0 'Admission' marked as sentence start (span begin)
2026-05-12 20:53:37.996 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=8] [doc 0] Token 96 'Per' marked as sentence start (span end next token)
2026-05-12 20:53:37.996 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=8] [doc 0] Token 96 'Per' marked as sentence start (span begin)
2026-05-12 20:53:37.996 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=8] [doc 0] Token 120 'The' marked as sentence start (span end next token)
2026-05-12 20:53:37.996 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=8] [doc 0] Token 120 'The' marked as sentence start (span begin)
2026-05-12 20:53:37.996 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=8] [doc 0] Token 128 'Vit

Locked Facts: ACE inhibitor, Amiodarone, Aortic stenosis, Apathy, Aspirin, Atrial fibrillation, Captopril, Coronary artery disease, Coumadin, Diltiazem, Endocarditis, Flagyl, Hypercholesterolemia, Hypertension, Iron, Lansoprazole, Lasix, Lescol, Levaquin, Levaquin allergy, Levofloxacin, Lopressor, Miconazole, Mitral regurgitation, Mitral valve rupture, Multivitamin, Nystatin, Pravastatin, Ritalin, Ruptured mitral valve chordae, Supraventricular tachycardia, Trazodone, Tylenol, Valsartan, Vancomycin, Zoloft, a drug reaction, abdominal pain, active bowel sounds, afterload reduction, alloantibodies, amiodarone, angiotensin, aortic stenosis, aortic valve replacement, apathetic syndrome, apathy, aspirin, atrial fibrillation, blood transfusions, captopril, cataract, cerebrovascular accidents, chest pain, chills, chronic lacunar infarctions, constipation, corona radiata, right vasoganglia, coronary artery disease, cough, depressive, diarrhea, diltiazem, disposition, dry, embolic cerebrovascul